In [2]:
import rawpy
import numpy as np
from PIL import Image

# Common rawpy encoding:
# 0=R, 1=G, 2=B, 3=G
RGGB = np.array([[0, 1],
                 [3, 2]], dtype=np.uint8)

def normalize_to_rggb(mosaic, pattern):
    transforms = [
        ("none", lambda x: x, lambda p: p),
        ("rot90", lambda x: np.rot90(x, 1), lambda p: np.rot90(p, 1)),
        ("rot180", lambda x: np.rot90(x, 2), lambda p: np.rot90(p, 2)),
        ("rot270", lambda x: np.rot90(x, 3), lambda p: np.rot90(p, 3)),
        ("flipud", np.flipud, np.flipud),
        ("fliplr", np.fliplr, np.fliplr),
        ("flipud_rot90", lambda x: np.rot90(np.flipud(x), 1),
                         lambda p: np.rot90(np.flipud(p), 1)),
        ("fliplr_rot90", lambda x: np.rot90(np.fliplr(x), 1),
                         lambda p: np.rot90(np.fliplr(p), 1)),
    ]

    for name, img_fn, pat_fn in transforms:
        if np.array_equal(pat_fn(pattern), RGGB):
            return img_fn(mosaic), name

    raise ValueError(f"Cannot normalize this CFA pattern to RGGB: {pattern}")

def save_cr3_as_rggb_png(cr3_path, png_path, normalize_black=True):
    with rawpy.imread(cr3_path) as raw:
        # Get the visible region's original Bayer data
        mosaic = raw.raw_image_visible.copy()
        pattern = raw.raw_pattern

        if pattern is None:
            raise RuntimeError("This is not a standard Bayer RAW, cannot process as RGGB directly.")

        # Normalize to RGGB
        mosaic, tfm = normalize_to_rggb(mosaic, pattern)

        # Convert to uint16, and optionally normalize black level
        mosaic = mosaic.astype(np.uint16)

        if normalize_black:
            black = min(raw.black_level_per_channel)
            white = raw.white_level

            mosaic = np.clip(mosaic.astype(np.int32) - int(black), 0, None)
            denom = max(1, int(white) - int(black))
            mosaic = (mosaic * 65535 // denom).astype(np.uint16)

        # Save as 16-bit single channel PNG
        img = Image.fromarray(mosaic)
        img.save(png_path)

        print("Original pattern:", pattern)
        print("Normalized to RGGB, transform:", tfm)
        print("Output:", png_path)
        print("shape:", mosaic.shape, "dtype:", mosaic.dtype)

# Example
save_cr3_as_rggb_png("input_photo/IMG_3203.CR3", "input_photo/IMG_3203.png")

Original pattern: [[0 1]
 [3 2]]
Normalized to RGGB, transform: none
Output: input_photo/IMG_3203.png
shape: (4020, 6024) dtype: uint16
